# 04 · Construct registry & gene order (Stage 6)

Register the chosen panel, run pre-synthesis sequence checks (free
cysteines, internal restriction sites, frame), and archive the full
table to CSV — *including the denominator*, per the reproducibility
checklist.

In [1]:
import sys, os
# make the package importable from the notebooks/ directory
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
EXAMPLES = os.path.join(ROOT, "examples")
print("project root:", ROOT)


project root: /home/user/biofx_python/enzyme_design


In [2]:
from enzyme_design.registry import (ConstructRegistry, DesignRecord,
                                     restriction_sites, free_cysteines)
reg = ConstructRegistry()
reg.add(DesignRecord('design_0006', protein_seq='MAGYSTVKDEFHIKLNPQRWG',
                     catalytic_residues=['A84','A85','A86','A87'],
                     cluster='A', score=0.05,
                     dna_seq='ATGGCTGGTTATAGTACCGTTAAA'))
reg.add(DesignRecord('design_0001', protein_seq='MAGCSTVKDEFHIKLNPQRWG',
                     catalytic_residues=['A84','A85','A86','A87'],
                     cluster='A', score=0.11,
                     dna_seq='ATGGCTTGCAGCACCCATATGAAA'))  # has Cys + NdeI
print('registry size:', len(reg))

registry size: 2


## Pre-synthesis warnings

In [3]:
for rec in reg:
    print(rec.design_id, '->', rec.pre_synthesis_warnings() or 'clean')

design_0006 -> clean
design_0001 -> ['free cysteine(s) at [4] (remove unless functional)', "internal restriction site(s): {'NdeI': [15]}"]


## Spot-check the helpers

In [4]:
print('free cysteines in MAGCSTV...:', free_cysteines('MAGCSTVKDEFHIKLNPQRWG'))
print('restriction sites:', restriction_sites('ATGGCTTGCAGCACCCATATGAAA'))

free cysteines in MAGCSTV...: [4]
restriction sites: {'NdeI': [15]}


## Archive to CSV and reload

In [5]:
reg.to_csv('panel_registry.csv')
reloaded = ConstructRegistry.from_csv('panel_registry.csv')
print('reloaded:', len(reloaded), 'records')
r = reloaded.get('design_0006')
print('catalytic residues:', r.catalytic_residues, ' score:', r.score)
print(open('panel_registry.csv').read())

reloaded: 2 records
catalytic residues: ['A84', 'A85', 'A86', 'A87']  score: 0.05
design_id,protein_seq,catalytic_residues,cluster,score,motif_ca_rmsd,plddt,worst_preorg,dna_seq,notes
design_0006,MAGYSTVKDEFHIKLNPQRWG,A84;A85;A86;A87,A,0.05,,,,ATGGCTGGTTATAGTACCGTTAAA,
design_0001,MAGCSTVKDEFHIKLNPQRWG,A84;A85;A86;A87,A,0.11,,,,ATGGCTTGCAGCACCCATATGAAA,

